# Machine Learning Foundations

This notebook focuses on:
 1. Features, Targets, Leakage
 2. Train / Validation / Test Splitting
 3. Classification (all major families ecept Neural Networks, we do those in a seperate Notebook)
 4. Optimization
 5. Decision Boundaries Visualization


In [ ]:

# DATA LOADING 
# WHAT: Load dataset
# WHY: We need a real dataset (not synthetic toys anymore)
# WHY IRIS: simple, interpretable, real measurements

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris

data = load_iris()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print(X.head())
print(y.head())


## What is a Feature?

A feature is a **measurable property of an object**.

In this dataset:
- sepal length
- sepal width
- petal length
- petal width

These are **experimental measurements**.

## What is the Target?

The target is the **variable we want to predict**.

Here:
- flower species (0, 1, 2)


## Important Insight

ML does NOT work with objects:
- not "flowers"
- not "molecules"

It works with:
--> numbers (feature vectors)


## Data Leakage

Definition:
The model has access to information that will not be available at prediction time.

## Why this breaks everything

Model learns:

"if leak == 1 -->  class = 1"

That’s not learning -->  it’s cheating.


## Real-world examples

- Using future measurements
- Using aggregated dataset statistics
- Using labels encoded in features


## Exercise
Give an example of leakage in a chemical assay dataset.

In [ ]:
# DATA LEAKAGE 
# WHAT: show incorrect modeling
# WHY: this is the #1 real-world failure

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_leak = X.copy()

# WRONG: target added as feature
X_leak["leak"] = y

X_train, X_test, y_train, y_test = train_test_split(X_leak, y)

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Leak accuracy:", accuracy_score(y_test, preds))

## Why splitting matters

Train --> learn parameters  
Validation --> choose model  
Test --> estimate final performance  

## Key rule

Test data must remain untouched.

## Failure modes

- Using test data during tuning -->  overestimation
- Reusing validation repeatedly -->  hidden leakage

## Train / Validation / Test Splitting

### 1. Problem

We want to evaluate how well a model works on **new data**.

But:

We only have one dataset.

### 2. Manual Solution

Use the same data for:

- training
- evaluation

### 3. Why This Fails

The model can memorize the data.

Result:

- training accuracy -->  very high
- real-world performance -->  unknown


### 4. Solution

Split the dataset into:

- Train -->  used to learn
- Validation -->  used to tune model
- Test -->  used for final evaluation


### 5. Intuition

We simulate time:

- Train = past data
- Validation = recent data
- Test = future data


### 6. Important Rules

- Test data must NEVER be used in training
- Validation data must NOT leak into training
- Splits must represent real-world usage


### 7. Scientific Interpretation

Each row is an experiment.

We want to test:

    "Will this model work on new experiments?"

So we must simulate new experiments.


### 8. Takeaway

Splitting =

    creating a realistic evaluation scenario

In [ ]:
# TRAIN / VALIDATION / TEST SPLIT

# WHAT:
# Split dataset into three parts:
# - training set
# - validation set
# - test set

# WHY:
# We want to simulate "future unseen data"
# The model must be evaluated on data it has never seen


# STEP 1 - TRAIN vs TEMP

# We first split dataset into:

# 60% training
# 40% temporary (will later become validation + test)


X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4)


# STEP 2 - VALIDATION vs TEST

# Split the remaining 40% into:
# 20% validation
# 20% test


X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
# VISUALIZE DATA 
# WHAT: plot feature space
# WHY: understand structure BEFORE modeling

plt.scatter(X.iloc[:, 0], X.iloc[:, 1], c=y)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Iris Dataset (first 2 features)")
plt.show()

## Decision Boundary -- How Models Separate Classes


### 1. Core Idea

A decision boundary is:

    the line (or surface) that separates different classes

Example:

    one side -->  class A  
    other side -->  class B  


### 2. Role in Classification

Every classifier creates a boundary:

    f(x) --> class

The boundary is where the model is **uncertain** between classes.


### 3. Geometry

Different models create different boundaries:

- Linear --> straight lines  
- Tree --> rectangular regions  
- KNN --> irregular, local shapes  
- SVM --> smooth curves  


### 4. Intuition

The model partitions the feature space into regions:

    each region -->  one class

So classification =

    dividing space into labeled zones


### 5. Why It Matters

The boundary determines:

- generalization to new data  
- sensitivity to noise  
- model behavior  


### 6. Takeaway

Decision boundary =

    how a model divides feature space into classes


In [ ]:
# DECISION BOUNDARY
# WHAT: visualize how models partition space

def plot_boundary(model, X, y):
    import numpy as np

    X_np = X.values
    X2 = X_np[:, :2]  # use only 2 features

    xx, yy = np.meshgrid(
        np.linspace(X2[:, 0].min(), X2[:, 0].max(), 200),
        np.linspace(X2[:, 1].min(), X2[:, 1].max(), 200)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]

    # pad remaining features
    full = np.zeros((grid.shape[0], X_np.shape[1]))
    full[:, :2] = grid

    Z = model.predict(full).reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.3)
    plt.scatter(X2[:, 0], X2[:, 1], c=y)
    plt.show()


## Linear Classifier -- Straight-Line Separation

### 1. Core Idea

A linear classifier separates classes using a straight boundary:

    w · x + b = 0

This creates a hyperplane (line in 2D).


### 2. How It Works

The model computes:

    score = w · x + b

Then decides class based on the sign or probability:

- positive -->  class A  
- negative -->  class B  


### 3. Intuition

It tries to draw a straight line (or plane) that best separates the data.

If the data is linearly separable:

--> works very well  


### 4. Limitations

If the pattern is nonlinear:

--> cannot separate classes correctly  

Example:

    circular or XOR patterns

### 5. Decision Boundary

Produces:

--> straight lines (or flat planes in higher dimensions)

### 6. Types of Linear Classifiers

- Logistic Regression --> probabilities  
- Perceptron --> simple rule-based updates  
- Linear SVM --> margin optimization  


### 7. Strengths

- simple and interpretable  
- fast to train  
- works well for high-dimensional data  


### 8. Weaknesses

- cannot model nonlinear relationships  
- limited flexibility  

### 9. Takeaway

Linear Classifier =

    straight boundary in feature space


In [ ]:
# LINEAR CLASSIFIER
# Idea: draw linear boundary

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

plot_boundary(model, X, y)

## Support Vector Machine (SVM) -- Margin-Based Classification
### 1. Core Idea

Instead of just separating classes:

--> Find the boundary that maximizes the distance between classes.

This distance is called the **margin**.

### 2. How It Works

SVM looks for a decision boundary such that:

- points of different classes are separated  
- the distance to the nearest points is maximized  

Only a few points matter:
--> the **support vectors** (closest to the boundary)

### 3. Intuition

Many boundaries can separate data.

SVM chooses:

--> the **most stable** one (largest margin)

Why?

--> better generalization to new data

### 4. Nonlinear Extension

If data is not linearly separable:

--> SVM uses a **kernel trick**

This maps data into a higher-dimensional space:

--> where it becomes separable

### 5. Decision Boundary

SVM creates:

- straight boundary → linear kernel  
- smooth curved boundary → RBF kernel  

Usually:

--> very clean and smooth separation regions

### 6. Strengths

- works well for small datasets  
- robust to overfitting  
- strong theoretical foundation  

### 7. Weaknesses

- sensitive to parameter tuning  
- slow on large datasets  
- less interpretable  

### 8. Takeaway

SVM =

    find boundary with maximum margin

In [ ]:
# SUPPORT VECTOR MACHINE 
# Idea: maximize margin

from sklearn.svm import SVC

model = SVC(kernel="rbf")
model.fit(X_train, y_train)

plot_boundary(model, X, y)


## Decision Tree -- Rule-Based Classification

### 1. Core Idea

Split the data step-by-step using simple rules:

    if feature > threshold --> go right
    else --> go left

Each path leads to a final prediction.

### 2. How It Works

The model builds a tree:

1. Choose the best feature to split the data  
2. Divide data into subsets  
3. Repeat recursively  

Until stopping condition is reached.


### 3. Splitting Criterion

At each step, the tree chooses a split that best separates classes.

Common methods:
- Gini impurity  
- Entropy  

Goal:
--> make groups as “pure” as possible


### 4. Intuition

The model is basically:

    a sequence of if–else rules

So it directly partitions feature space into regions.

### 5. Decision Boundary

Creates:

--> rectangular, axis-aligned regions  

Each split cuts the space into smaller boxes.

### 6. Strengths

- Easy to understand  
- Captures nonlinear relationships  
- Works well on tabular data  

### 7. Weaknesses

- Overfits easily  
- Very sensitive to small data changes  
- Unstable as a single model  

### 8. Takeaway

Decision Tree =

    recursive splitting of feature space using rules

In [ ]:
# DECISION TREE 
# Idea: recursive splitting

from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=4)
model.fit(X_train, y_train)

plot_boundary(model, X, y)


## Random Forest -- Ensemble of Decision Trees

### 1. Core Idea

Instead of using a single decision tree:

--> Use many trees and combine their predictions.

Each tree gives a vote, and the final prediction is:

    majority vote (classification)

### 2. Why This Is Needed

A single decision tree:

- is very flexible
- but **overfits easily**

Random Forest solves this by:

--> averaging many trees

### 3. How It Works

For each tree:

1. Train on a random subset of the data (sampling with replacement)  
2. Use only a random subset of features at each split  

Then:

--> combine predictions from all trees

### 4. Intuition

Each tree is slightly different.

Individually:
--> noisy and unstable

Together:
--> errors cancel out  
--> predictions become stable


### 5. Decision Boundary

Random Forest creates:

--> smoother and more robust regions  
compared to a single tree

But still captures nonlinear patterns.

### 6. Strengths

- Handles nonlinear data well  
- Reduces overfitting compared to single trees  
- Works well on tabular data  

### 7. Weaknesses

- Less interpretable than a single tree  
- Can be slower with many trees  
- Does not extrapolate beyond training data  


### 8. Takeaway

Random Forest =

    many decision trees + averaging


In [ ]:
# RANDOM FOREST
# Idea: combine many trees

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

plot_boundary(model, X, y)


## K-Nearest Neighbors (KNN) -- Distance-Based Classification

### 1. Core Idea

To classify a new data point:

    "Look at nearby data points and take a vote."

--> The predicted class is the most common class among the **k nearest neighbors**.


### 2. How It Works

For a new sample:

1. Compute distance to all training points  
2. Select the k closest  
3. Assign the majority class  


### 3. Distance

Closeness is usually measured using:

    Euclidean distance

So:

--> similar measurements = close in space

### 4. Key Parameter: k

- small k → very sensitive → overfitting  
- large k → smoother decisions → may underfit  

### 5. Intuition

KNN does not learn a model.

Instead:

--> it compares new data to stored examples  

So it is a **memory-based method**.

### 6. Decision Boundary

KNN creates:

--> local, irregular regions  

because prediction depends on nearby points.


### 7. Important Insight

Feature scaling matters:

If one feature has larger values → it dominates distance.


### 8. Takeaway

KNN =

    distance calculation + neighbor voting


In [ ]:
# KNN 
# Idea: classify based on neighbors

from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

plot_boundary(model, X, y)



## Naive Bayes -- Probabilistic Classification


### 1. Problem

We want to classify data:

    Given features x → predict class y

Example:
Measurements of a flower → predict species


### 2. Core Idea

Instead of learning a boundary directly (like trees or SVM),

Naive Bayes asks:

    How likely is this data point for each class?

It computes:

    P(class | features)

and chooses the class with the highest probability.


### 3. Bayes’ Theorem

Naive Bayes is based on:

    P(class | features) =
        P(features | class) * P(class)
        --------------------------------
              P(features)

For classification, we compare:

    P(features | class) * P(class)


### 4. The "Naive" Assumption

The difficult term is:

    P(x1, x2, ..., xd | class)

Naive Bayes assumes:

    Features are independent

So it simplifies to:

    P(features | class) =
        P(x1|class) * P(x2|class) * ... * P(xd|class)


### 5. Why This Works (Even Though It’s Wrong)

Features are usually NOT independent (e.g. petal length and width).

But:

- we only need relative probabilities
- strong signals dominate
- errors partially cancel out

--> surprisingly effective in practice, Magic **


### 6. Gaussian Naive Bayes (used here,m there are other variants)

Assumes each feature follows a normal distribution:

    P(x | class) = Normal(mean, variance)

So the model learns:

- mean per feature per class
- variance per feature per class


### 7. Intuition

Naive Bayes does NOT explicitly draw boundaries.

Instead, it asks:

    "Under which class is this point most likely?"

Decision boundary = where probabilities are equal.


### 8. Strengths and Weaknesses

**Strengths**
- very fast
- works well with small datasets
- robust to noise

**Weaknesses**
- independence assumption unrealistic
- cannot model interactions between features

In [ ]:
# NAIVE BAYES
# Idea: probability-based classification

from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(X_train, y_train)

plot_boundary(model, X, y)

## Decision Boundaries -- Visualizing Model Behavior

### 1. Core Idea

A decision boundary is:

    the region where a model switches from one class to another

It is defined by the model:

    f(x) --> class


### 2. Why We Visualize It

Without visualization:

--> models are abstract

With visualization:

--> we see how the model "thinks"


### 3. Interpretation

Each model creates different boundaries:

- Linear models --> straight lines  
- Trees --> rectangular regions  
- KNN --> irregular local shapes  
- SVM --> smooth curved boundaries  

### 4. Connection to Data

The boundary depends on:

- data distribution  
- noise  
- model assumptions  

### 5. Key Insight

Classification =

    dividing feature space into labeled regions

### 6. What To Look For

When analyzing plots:

- Is boundary smooth or complex?  
- Does it follow data or noise?  
- Does it generalize beyond training points?  

### 7. Takeaway

Decision boundaries reveal:

    how a model represents the data

## Optimization -- How Models Learn

### 1. Core Idea

All models try to:

    minimize error between prediction and truth

This is done using a **loss function**.


### 2. Loss Function

For classification:

    Cross-Entropy Loss

Measures:

- how wrong the prediction is  
- how confident the model is  


### 3. Optimization Goal

Find parameters (weights) that minimize loss:

    best model = lowest error


### 4. Gradient Descent

The most common method:

    parameter = parameter - learning_rate * gradient

- gradient = direction of increasing error  
- we move opposite → reduce error  


### 5. Intuition

Think of a landscape:

- high points = bad model  
- low points = good model  

Gradient descent:

--> walk downhill


### 6. Not All Models Use It

- Linear models --> gradient descent  
- Neural networks -->  gradient descent  
- Trees / Random Forest --> greedy splitting  
- KNN --> no optimization (just lookup)  


### 7. Why It Matters

Optimization determines:

- how fast the model learns  
- if it converges  
- if it gets stuck  

### 8. Takeaway

Optimization =

    process of improving model parameters to reduce error


## Summary of the Optimazation


| Model Family        | Optimization Type                     | How It Works                                      |
|--------------------|--------------------------------------|---------------------------------------------------|
| Linear Models      | Gradient Descent                     | Iteratively update weights to minimize loss       |
| Neural Networks    | Gradient Descent + Backpropagation   | Compute gradients through layers using chain rule |
| SVM                | Convex Optimization                 | Solve margin-maximization problem (QP solver)     |
| Decision Tree      | Greedy Splitting                     | Choose best split at each step (local optimum)    |
| Random Forest      | Multiple Trees + Averaging           | No global optimization, reduce error via averaging|
| KNN                | None                                 | No training, just distance-based lookup           |
| Naive Bayes        | Direct Statistical Estimation        | Estimate probabilities from data (no iteration)   |

### Key Insight

"Training a model" does NOT always mean gradient descent.

Different models use fundamentally different strategies to learn.

In [ ]:
# COMPUTING LOSS FOR DIFFERENT CLASSIFIERS

# WHAT:
# Compare models using a COMMON loss (cross-entropy)

# WHY:
# Even if models use different training methods,
# we can evaluate them with the same loss

from sklearn.metrics import log_loss

# Helper function

def evaluate_model_loss(model, X_train, y_train, X_test, y_test):
    """
    Trains model and computes log loss + accuracy
    """

    # Train model
    model.fit(X_train, y_train)

    # Try to get probabilities
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_test)
    else:
        # fallback for models without probabilities
        preds = model.predict(X_test)
        
        # convert to fake probabilities (not ideal)
        probs = np.zeros((len(preds), len(np.unique(y_test))))
        probs[np.arange(len(preds)), preds] = 1

    # Compute loss
    loss = log_loss(y_test, probs)

    return loss

# TEST DIFFERENT MODELS

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Decision Tree": DecisionTreeClassifier(max_depth=4),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(5),
    "SVM": SVC(probability=True),  # enable probabilities
    "Naive Bayes": GaussianNB()
}

for name, model in models.items():
    loss = evaluate_model_loss(model, X_train, y_train, X_test, y_test)
    print(f"{name:20} → Log Loss: {loss:.4f}")

## Training Loss vs Evaluation Metric

### 1. Core Idea

There are two different concepts:

- **Training Loss** → used during learning  
- **Evaluation Metric** → used after training  

They answer different questions.

### 2. Training Loss

**Definition:**

Training loss is the function a model tries to minimize during training.

Example:
- Cross-entropy (classification)
- Mean squared error (regression)
- Hinge loss (SVM)

### What it does

During training:

    model --> predicts → compute loss → update parameters

So training loss guides:

    How should the model change to improve?